In [1]:
!pip install transformers==4.41.2
!pip install datasets==2.20
!pip install torch==2.3.1
!pip install peft==0.11.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 27.5 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.0
    Uninstalling tokenizers-0.21.0:
      Successfully uninstalled tokenizers-0.21.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.47.1
    Uninstalling transformers-4.47.1:
      Successfully uninstalled transformers-4.47.1
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 547.8/547.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.1/316.1 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

In [3]:
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, GenerationConfig, TrainingArguments, Trainer
import torch
import time
#import evaluate  ## for calculating rouge score
import pandas as pd
import numpy as np

# Loading dataset

In [4]:
## dataset from https://huggingface.co/datasets/knkarthick/dialogsum
huggingface_dataset_name = "knkarthick/dialogsum"

dataset = load_dataset(huggingface_dataset_name)

dataset

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/12460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

In [5]:
dataset["train"][0]

{'id': 'train_0',
 'dialogue': "#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?\n#Person2#: I found it would be a good idea to get a check-up.\n#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.\n#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?\n#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.\n#Person2#: Ok.\n#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?\n#Person2#: Yes.\n#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.\n#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.\n#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.\n#Person2#: Ok, thanks doctor.",
 'summary': "Mr. Smith'

# Subset of Main Data

In [6]:
from datasets import load_dataset, DatasetDict

# Load your original dataset
#dataset = load_dataset('your_dataset_name')  # replace 'your_dataset_name' with the actual dataset name

# Select the specified number of examples from each split
train_subset = dataset['train'].select(range(30))
validation_subset = dataset['validation'].select(range(5))
test_subset = dataset['test'].select(range(5))

# Create a new DatasetDict with the smaller subsets
small_dataset = DatasetDict({
    'train': train_subset,
    'validation': validation_subset,
    'test': test_subset
})

small_dataset["train"][0]

{'id': 'train_0',
 'dialogue': "#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?\n#Person2#: I found it would be a good idea to get a check-up.\n#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.\n#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?\n#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.\n#Person2#: Ok.\n#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?\n#Person2#: Yes.\n#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.\n#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.\n#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.\n#Person2#: Ok, thanks doctor.",
 'summary': "Mr. Smith'

# Load LLM

In [7]:
## for more information of model https://huggingface.co/google/flan-t5-base
model_name = 'google/flan-t5-base'
#model_name = 't5-small'
device = "cuda" if torch.cuda.is_available() else "cpu"
# bfloat16 mean we are using the small version of flan-t5
original_model = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=torch.bfloat16)

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

In [9]:
original_model.to(device)

print(f"Model loaded on: {device}")

Model loaded on: cuda


# Trainable Parameter

In [10]:
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    return f'trainable model parameters: {trainable_model_params}\n \
            all model parameters: {all_model_params} \n \
            percentage of trainable model parameters: {(trainable_model_params / all_model_params) * 100} %'


print(print_number_of_trainable_model_parameters(original_model))

trainable model parameters: 247577856
             all model parameters: 247577856 
             percentage of trainable model parameters: 100.0 %


# data preprocessing

In [11]:
import torch

def tokenize_function(example):
    start_prompt = 'Summarize the following conversation. \n\n'
    end_prompt = '\n\nSummary: '

    # Construct the prompts for each dialogue in the batch
    prompts = [start_prompt + dialogue + end_prompt for dialogue in example["dialogue"]]

    # Tokenize inputs and labels
    input_encodings = tokenizer(prompts, padding='max_length', truncation=True, return_tensors='pt')
    label_encodings = tokenizer(example['summary'], padding='max_length', truncation=True, return_tensors='pt')

    # Move tensors to GPU if available
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    example['input_ids'] = input_encodings.input_ids.to(device)
    example['labels'] = label_encodings.input_ids.to(device)

    return example


# Optimize dataset mapping and column removal
tokenize_datasets = (
    small_dataset.map(tokenize_function, batched=True)  # Use batched=True for efficient processing
    .remove_columns(['id', 'topic', 'dialogue', 'summary'])  # Remove unnecessary columns
)


Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [12]:
tokenize_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 30
    })
    validation: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 5
    })
    test: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 5
    })
})

# USE PEFT Configuration

In [13]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(r=32, #rank 32,
                         lora_alpha=32, ## LoRA Scaling factor
                         target_modules=['q', 'v'], ## The modules(for example, attention blocks) to apply the LoRA update matrices.
                         lora_dropout = 0.05,
                         bias='none',
                         task_type=TaskType.SEQ_2_SEQ_LM ## flan-t5
)

## target_modules='q', This represents the value projection layer in the transformer model. The value projection layer transforms input tokens into value vectors,
# which are the actual values that are attended to based on the attention scores computed from query and key vectors.

## target_modules='v',This typically refers to the query projection layer in a transformer-based model. The query projection layer is responsible for transforming
# input tokens into query vectors, which are used to attend to other tokens in the sequence during self-attention mechanism.

In [14]:
peft_model = get_peft_model(original_model, lora_config)

print(print_number_of_trainable_model_parameters(peft_model))

trainable model parameters: 3538944
             all model parameters: 251116800 
             percentage of trainable model parameters: 1.4092820552029972 %


# Train-peft-model

In [15]:
#output_dir = f'./dialogue-summary-training-{str(int(time.time()))}'
## this is we are again back to the hugging face trainer module

peft_training_args = TrainingArguments(output_dir="trainer-api-logs-2025",
                                       auto_find_batch_size=True,
                                       evaluation_strategy="epoch",
                                       learning_rate=1e-3,
                                       num_train_epochs=1,
                                       logging_steps=1,
                                       max_steps=1,
                                       report_to='none'
                                       #load_best_model_at_end=True

                )

## this is same except we are using PEFT model instead of regular
peft_trainer = Trainer(model=peft_model,
                      args=peft_training_args,
                      train_dataset=tokenize_datasets['train'],
                      eval_dataset=tokenize_datasets['validation']
                 )


peft_trainer.train()

peft_model_path = 'peft-model-path-2025'

peft_trainer.model.save_pretrained(peft_model_path)
tokenizer.save_pretrained(peft_model_path)


/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
max_steps is given, it will override any value given in num_train_epochs


Epoch,Training Loss,Validation Loss
0,47.500000,51.000000


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


('peft-model-path-2025/tokenizer_config.json',
 'peft-model-path-2025/special_tokens_map.json',
 'peft-model-path-2025/spiece.model',
 'peft-model-path-2025/added_tokens.json',
 'peft-model-path-2025/tokenizer.json')

In [25]:
from peft import PeftModel
peft_model_base = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-base', torch_dtype=torch.bfloat16).to(device)
tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-base')

peft_model = PeftModel.from_pretrained(peft_model_base,
                                      'peft-model-path-2025',
                                      torch_dtype=torch.bfloat16,
                                      is_trainable=False).to(device) ## is_trainable mean just a forward pass jsut to get a sumamry

index = 9 ## randomly pick index
dialogue = dataset['test'][index]['dialogue']
human_baseline_summary = dataset['test'][index]['summary']

prompt = f"""
Summarize the following conversation.

{dialogue}

Summary:
"""

input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(device)

original_model_outputs = original_model.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=200, num_beams=1))
original_model_text_output = tokenizer.decode(original_model_outputs[0], skip_special_tokens=True)


peft_model_outputs = peft_model.generate(input_ids=input_ids, generation_config=GenerationConfig(max_new_tokens=200, num_beams=1))
peft_model_text_output = tokenizer.decode(peft_model_outputs[0], skip_special_tokens=True)

print(f'Human Baseline summary: \n{human_baseline_summary}\n')
print(f'Original Model Output \n{original_model_text_output}\n')
print(f'Peft Model Output \n{peft_model_text_output}\n')

Human Baseline summary: 
#Person1# and Brian are at the birthday party of Brian. Brian thinks #Person1# looks great and is popular.

Original Model Output 
Brian's birthday is coming up.nel is coming over to celebrate.nel is going to have a dance with Brian.nel is going to have a drink with Brian.nel is going to have a drink with Brian.nel is going to have a dance with Brian.nel is going to have a drink with Brian.nel is going to have a dance with Brian.nel is going to have a dance with Brian.nel is going to have a dance with Brian.nel is going to have a dance with Brian.nel is going to have a dance with Brian.nel is going to have a dance with Brian.nel is going to have a dance with Brian.nel is going to have a dance with Brian.

Peft Model Output 
Brian's birthday is coming up.nel is coming over to celebrate.nel is going to have a dance with Brian.nel is going to have a drink with Brian.nel is going to have a drink with Brian.nel is going to have a dance with Brian.nel is going to hav

In [23]:
 dataset['test'][12]

{'id': 'test_4_1',
 'dialogue': "#Person1#: This Olympic park is so big!\n#Person2#: Yes. Now we are in the Olympic stadium, the center of this park.\n#Person1#: Splendid! When is it gonna be finished?\n#Person2#: The whole stadium is to be finished this June.\n#Person1#: How many seats are there in the stand?\n#Person2#: Oh, there are 5000 seats in total.\n#Person1#: I didn ' t know it would be so big!\n#Person2#: It is! Look there, those are the tracks. And the jumping pit is over there.\n#Person1#: Ah... I see. Hey, look the sign here, No climbing.\n#Person2#: We put many signs with English translations for foreign visitors.",
 'summary': "#Person1# is surprised at the Olympic Stadium'volume, capacity and interior setting to #Person1#.",
 'topic': 'Olympic Stadium'}